# Participant-balanced NONAN pooling pilot

Corrects the window-dominance flaw found in the release and prior enrichment scripts. Each participant has equal mass within each source-by-label cell; candidate NONAN is additionally bounded to 10% or 25% of one native source-by-label cell. Participant-disjoint folds, fold-fitted normalization, and no frozen-cohort access are enforced.

In [1]:
from pathlib import Path
import sys,gc
import numpy as np,pandas as pd,torch
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score,balanced_accuracy_score
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P=ROOT/'data'/'processed'; N=ROOT/'data'/'interim'/'nonan_gaitprint'; D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',D)
x=np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'),np.load(P/'sint_maartenskliniek_external_windows_float32.npy')]); m=pd.concat([pd.read_csv(P/'validated_window_metadata.csv'),pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')],ignore_index=True); m=m[m.label.isin(['healthy','stroke'])].reset_index(drop=True); m['y']=m.label.eq('stroke').astype(int); m['source']=m.dataset_id; m['group']=m.participant_key.astype(str)
nx=np.load(N/'candidate_healthy_enrichment_magnitude_isolated_spike_repaired.npy',mmap_mode='r'); nm=pd.read_csv(N/'candidate_healthy_enrichment_window_metadata.csv'); nm['y']=0; nm['source']='nonan_gaitprint'; nm['group']=nm.participant_key.astype(str)
cap_rng=np.random.default_rng(42); keep=np.concatenate([cap_rng.choice(v,min(64,len(v)),replace=False) for v in nm.groupby('group').indices.values()]); nx=np.asarray(nx[keep]); nm=nm.iloc[keep].reset_index(drop=True)
people=pd.concat([m[['group','source','y']].drop_duplicates(),nm[['group','source','y']].drop_duplicates()],ignore_index=True); people['stratum']=people.source+'|'+people.y.astype(str)
def participant_cell_weights(frame,nonan_mass=0.):
    pcounts=frame.groupby('group').size(); cell_people=frame[['source','y','group']].drop_duplicates().groupby(['source','y']).size(); base=frame.group.map(1/pcounts)/pd.MultiIndex.from_frame(frame[['source','y']]).map(cell_people); mass=np.where(frame.source.eq('nonan_gaitprint'),nonan_mass,1.0); return torch.tensor((base*mass).to_numpy(),dtype=torch.double)
def score(net,arr,meta,mean,std):
    with torch.inference_mode(): p=torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
    g=meta.assign(p=p).groupby(['group','y'],as_index=False).p.mean(); return {'participants':len(g),'healthy':int((g.y==0).sum()),'stroke':int((g.y==1).sum()),'auroc':roc_auc_score(g.y,g.p) if g.y.nunique()==2 else np.nan,'balanced_accuracy':balanced_accuracy_score(g.y,g.p>=.5) if g.y.nunique()==2 else np.nan,'healthy_specificity':float((g.loc[g.y==0,'p']<.5).mean())}
rows=[]; folds=StratifiedKFold(3,shuffle=True,random_state=42)
for fold,(trp,vap) in enumerate(folds.split(people,people.stratum)):
    tg,vg=set(people.iloc[trp].group),set(people.iloc[vap].group); bt,bv=m.group.isin(tg).to_numpy(),m.group.isin(vg).to_numpy(); nt,nv=nm.group.isin(tg).to_numpy(),nm.group.isin(vg).to_numpy()
    for mode,mass in [('baseline_participant_balanced',0.),('nonan_participant_balanced_10pct',.10),('nonan_participant_balanced_25pct',.25)]:
        torch.manual_seed(42000+fold); tx,tm=x[bt],m.loc[bt].copy()
        if mass: tx,tm=np.concatenate([tx,nx[nt]]),pd.concat([tm,nm.loc[nt]],ignore_index=True)
        mean,std=tx.reshape(-1,3).mean(0),tx.reshape(-1,3).std(0).clip(1e-4); z=torch.from_numpy(((tx-mean)/std).transpose(0,2,1).astype('float32')); y=torch.from_numpy(tm.y.to_numpy('float32')); w=participant_cell_weights(tm,mass); dl=DataLoader(TensorDataset(z,y),128,sampler=WeightedRandomSampler(w,len(w),replacement=True,generator=torch.Generator().manual_seed(900+fold)))
        net=StrokeGaitInception().to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
        for _ in range(8):
            net.train()
            for a,b in dl: opt.zero_grad(); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)),b.to(D)); loss.backward(); opt.step()
        net.eval()
        for name,ex,em in [('original',x[bv],m.loc[bv]),('nonan_holdout',nx[nv],nm.loc[nv])]: rows.append({'fold':fold,'mode':mode,'evaluation':name,**score(net,ex,em,mean,std)})
        del net,opt,dl,z,y; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',fold,mode)
out=pd.DataFrame(rows); out.to_csv(P/'participant_balanced_nonan_pooling_pilot.csv',index=False); print(out.groupby(['mode','evaluation'])[['auroc','balanced_accuracy','healthy_specificity']].mean())

device: cuda


complete 0 baseline_participant_balanced


complete 0 nonan_participant_balanced_10pct


complete 0 nonan_participant_balanced_25pct


complete 1 baseline_participant_balanced


complete 1 nonan_participant_balanced_10pct


complete 1 nonan_participant_balanced_25pct


complete 2 baseline_participant_balanced


complete 2 nonan_participant_balanced_10pct


complete 2 nonan_participant_balanced_25pct
                                                   auroc  balanced_accuracy  \
mode                             evaluation                                   
baseline_participant_balanced    nonan_holdout       NaN                NaN   
                                 original       0.944290           0.846033   
nonan_participant_balanced_10pct nonan_holdout       NaN                NaN   
                                 original       0.948641           0.873769   
nonan_participant_balanced_25pct nonan_holdout       NaN                NaN   
                                 original       0.959078           0.873131   

                                                healthy_specificity  
mode                             evaluation                          
baseline_participant_balanced    nonan_holdout             0.787274  
                                 original                  0.723811  
nonan_participant_balanced_10pct nonan_hold

Only a candidate arm that improves held-out candidate healthy specificity without lowering original-source balanced accuracy or healthy specificity can advance to a repeated paired gate. This isolates sampling correction; no segmentation, feature, architecture, threshold, or frozen-test change is made here.